---
🟢 **Level 1 — Basic DataFrame coding**

- Read CSV, JSON and Parquet files using PySpark. 🟢
- Read a CSV using an explicit schema. 🟢
- Select specific columns from a DataFrame. 🟢
- Rename multiple columns. 🟢
- Add a new column using `withColumn()`. 🟢
- Apply multiple conditions using `when()` / `otherwise()`. 
- Filter records based on multiple conditions. 🟢
- Convert a string column to date/timestamp.🟢
- Handle NULL values using `fillna()`, `dropna()` and `when()`.🟢
- Remove duplicate records. 🟢
- Count total rows and distinct values. 🟢
- Find distinct customers. 🟢
- Sort a DataFrame by multiple columns. 🟢
- Convert a column to uppercase/lowercase. 🟢
- Extract year, month and day from a date.🟢
- Concatenate multiple columns.🟢
- Split a string column.🟢
- Extract data using regular expressions.🟢
- Create a column based on multiple business conditions.🟢
- Rename columns dynamically. 🟢
---

In [0]:
#Read CSV, JSON and Parquet files using PySpark.

from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

df = spark.read.format("csv")\
    .option("header",True)\
    .option("inferSchema",True)\
    .load("/Volumes/workspace/default/practice_files/departments.csv")
df.show()

df1 = spark.read.format("json")\
    .option("inferSchema",True)\
    .option("multiline",True)\
    .load("/Volumes/workspace/default/practice_files/employees.json")
df1.show()

df2 = spark.read.format("parquet")\
    .load("/Volumes/workspace/default/practice_files/employees_parquet")
df2.show()

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

#Select specific columns from a DataFrame. --
#Rename multiple columns. --
#Filter records based on multiple conditions. --
#Remove duplicate records.--
#Count total rows and distinct values. --
#Find distinct customers. --
#Sort a DataFrame by multiple columns. --
#Convert a column to uppercase/lowercase. --
#Add a new column using withColumn()

df3 = df2.select(col("EmployeeID").alias("ID"),col("Name").alias("ename"),"Department","Salary","Age","City","DepartmentID")\
    .filter((col("Salary")>50000) &(col("age")>25))
df3.show()
drop_duplicates_df = df3.dropDuplicates(["ID","ename"])
count_df = drop_duplicates_df.select("ID","ename","Department","Salary","DepartmentID","Age").distinct().count()
print(count_df)
distinct_ename_df = df3.select("ID","ename").distinct()
distinct_ename_df.show()
short_byMultiple_df = df3.orderBy(desc("Salary"),desc("DepartmentID"))
short_byMultiple_df.show()
Upper_Lower_df = df3.select(upper(col("ename")).alias("ENAME"),lower(col("City")).alias("city"))
Upper_Lower_df = Upper_Lower_df.withColumn("Active",lit("Y"))
Upper_Lower_df.show()

In [0]:
sample_data = [
    (1, "John Doe", "2025-07-15", "CA,USA", "M123456", 35000, None),
    (2, "Jane Smith", "2026-01-25", "NY,USA", "F234567", 50000, 29),
    (3, "Jenny Lee", None, "TX,USA", "M345678", None, 31),
    (4, "Mike Brown", "2026-08-18", "CA,USA", "F456789", 62000, 35),
    (5, "Ann Kim", "2026-03-01", "FL,USA", "", 70000, 27)
]

sample_schema = StructType([
    StructField("ID", IntegerType(), True),
    StructField("Name", StringType(), True),
    StructField("HireDate", StringType(), True),
    StructField("Location", StringType(), True),
    StructField("EmpCode", StringType(), True),
    StructField("Salary", IntegerType(), True),
    StructField("Age", IntegerType(), True)
])

sample_df = spark.createDataFrame(sample_data, sample_schema)
display(sample_df)

In [0]:
# Extract year, month and day from a date.
df_date = sample_df.withColumn("HireDateDT", to_date(col("HireDate"), "yyyy-MM-dd"))\
    .withColumn("Year", year(col("HireDateDT")))\
    .withColumn("Month", month(col("HireDateDT")))\
    .withColumn("Day", dayofmonth(col("HireDateDT")))
df_date.show()

# Concatenate multiple columns.
df_concat = sample_df.select(concat(col("Name"),col("Location")).alias("Name_and_location")).show()
df_concat_ws = sample_df.select(concat_ws(" | ",col("Name"),col("Location")).alias("Name_and_location")).show()

df_concat_withColumn = sample_df.withColumn("Name_and_location",concat(col("Name"),col("Location"))).show()
df_concat_ws_withColumn = sample_df.withColumn("Name_and_location",concat_ws(" | ",col("Name"),col("Location"))).show()


# Split a string column.
sample_df_split = sample_df.withColumn("State", split(col("Location"), ",")[0]) \
    .withColumn("Country", split(col("Location"), ",")[1])
display(sample_df_split)

# Extract data using regular expressions.
sample_df_regex = sample_df.withColumn("LetterCode", regexp_extract(col("EmpCode"), r"([A-Z])\d+", 1))
display(sample_df_regex)

# Create a column based on multiple business conditions.
sample_df_conditions = sample_df.withColumn("Seniority", when(col("Salary") >= 60000, "Senior")
                                           .when((col("Salary") >= 40000) & (col("Salary") < 60000), "Mid")
                                           .otherwise("Junior"))
display(sample_df_conditions)

# Apply multiple conditions using when() / otherwise().
sample_df_apply_when = sample_df.withColumn("Status", when(col("HireDate").isNull(), "No Hire Date")
                                            .when(col("Age").isNull(), "No Age")
                                            .otherwise("Complete"))
display(sample_df_apply_when)
,
# Convert a string column to date/timestamp.
sample_df_to_timestamp = sample_df.withColumn("HireDate_ts", to_timestamp(col("HireDate"), "yyyy-MM-dd"))
display(sample_df_to_timestamp)

# Handle NULL values using fillna(), dropna() and when().
sample_df_fillna = sample_df.fillna({"HireDate": "2000-01-01", "Salary": 0, "Age": 0})
display(sample_df_fillna)
sample_df_dropna = sample_df.dropna(subset=["HireDate", "Salary"])
display(sample_df_dropna)
sample_df_when_null = sample_df.withColumn("SalaryCategory", when(col("Salary").isNull(), "Unknown").otherwise("Known"))
display(sample_df_when_null)